## Day 2
Task 2

# Data Cleaning & KPI Creation

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\Priti\Downloads\defects_data.csv")   
df.head()

,defect_id,product_id,defect_type,defect_date,defect_location,severity,inspection_method,repair_cost
0,1,15,Structural,06-06-2024,Component,Minor,Visual Inspection,245.47
1,2,6,Functional,4/26/2024,Component,Minor,Visual Inspection,26.87
2,3,84,Structural,2/15/2024,Internal,Minor,Automated Testing,835.81
3,4,10,Functional,3/28/2024,Internal,Critical,Automated Testing,444.47
4,5,14,Cosmetic,4/26/2024,Component,Minor,Manual Testing,823.64


# Handle:
Missing values

In [3]:
df.isnull().sum()

defect_id            0
product_id           0
defect_type          0
defect_date          0
defect_location      0
severity             0
inspection_method    0
repair_cost          0
dtype: int64

#
Inconsistent defect names

In [4]:
# Convert to lowercase
df['defect_type'] = df['defect_type'].str.lower().str.strip()
df['defect_type']

0      structural
1      functional
2      structural
3      functional
4        cosmetic
          ...    
995    structural
996    functional
997    structural
998      cosmetic
999      cosmetic
Name: defect_type, Length: 1000, dtype: object

# 
Date formatting

In [5]:
df['defect_date'].head()

0    06-06-2024
1     4/26/2024
2     2/15/2024
3     3/28/2024
4     4/26/2024
Name: defect_date, dtype: object

In [6]:
df['defect_date'] = pd.to_datetime(df['defect_date'], format='mixed')
df['defect_date']

0     2024-06-06
1     2024-04-26
2     2024-02-15
3     2024-03-28
4     2024-04-26
         ...    
995   2024-03-01
996   2024-03-21
997   2024-01-16
998   2024-06-21
999   2024-03-23
Name: defect_date, Length: 1000, dtype: datetime64[ns]

## 
Remove Duplicates

In [7]:
df.duplicated().sum()

np.int64(0)

# Create derived metrics
Defect per Product Id 

In [8]:
product_wise_defects = df.groupby('product_id')['defect_id'].count()
print(product_wise_defects)

product_id
1      12
2       7
3       8
4      16
5      13
       ..
96     13
97     19
98      8
99     10
100     7
Name: defect_id, Length: 100, dtype: int64


In [10]:
# Product ID Wise Severity Count
product_severity = pd.crosstab(df['product_id'], df['severity'])
print(product_severity)

severity    Critical  Minor  Moderate
product_id                           
1                  8      3         1
2                  3      2         2
3                  2      1         5
4                  6      5         5
5                  6      2         5
...              ...    ...       ...
96                 5      6         2
97                 6      5         8
98                 4      1         3
99                 2      7         1
100                2      2         3

[100 rows x 3 columns]


In [13]:
# Best KPI
df.groupby('product_id')['defect_id'].count().sort_values(ascending=False)

product_id
81    20
63    20
97    19
56    18
4     16
      ..
19     4
89     4
52     4
85     4
55     3
Name: defect_id, Length: 100, dtype: int64

In [15]:
# Product ID Wise Defect Type Count
product_defect_type = pd.crosstab(df['product_id'], df['defect_type'])
print(product_defect_type)

defect_type  cosmetic  functional  structural
product_id                                   
1                   5           3           4
2                   2           4           1
3                   0           3           5
4                   7           5           4
5                   4           4           5
...               ...         ...         ...
96                  6           5           2
97                  9           5           5
98                  4           4           0
99                  5           3           2
100                 1           5           1

[100 rows x 3 columns]


In [16]:
# Product ID Wise Average Repair Cost
avg_repair_cost_product = df.groupby('product_id')['repair_cost'].mean()
print(avg_repair_cost_product)

product_id
1      554.382500
2      455.628571
3      388.331250
4      405.965625
5      454.808462
          ...    
96     456.094615
97     442.412632
98     575.547500
99     559.813000
100    624.592857
Name: repair_cost, Length: 100, dtype: float64


# 
Monthly defect count 

In [17]:
df['defect_date'] = pd.to_datetime(df['defect_date'], errors='coerce')
df['month'] = df['defect_date'].dt.to_period('M')
monthly_defects = (
    df.groupby(['month', 'defect_type'])
      .size()
      .reset_index(name='count')
)

print(monthly_defects)

      month defect_type  count
0   2024-01    cosmetic     65
1   2024-01  functional     56
2   2024-01  structural     70
3   2024-02    cosmetic     47
4   2024-02  functional     55
5   2024-02  structural     58
6   2024-03    cosmetic     46
7   2024-03  functional     67
8   2024-03  structural     62
9   2024-04    cosmetic     49
10  2024-04  functional     52
11  2024-04  structural     55
12  2024-05    cosmetic     57
13  2024-05  functional     66
14  2024-05  structural     44
15  2024-06    cosmetic     45
16  2024-06  functional     43
17  2024-06  structural     63


##  Create time-based columns: 

In [18]:
df['defect_date'].head()

0   2024-06-06
1   2024-04-26
2   2024-02-15
3   2024-03-28
4   2024-04-26
Name: defect_date, dtype: datetime64[ns]

# 
Month 

In [19]:
df['month'] = df['defect_date'].dt.month
df['month']

0      6
1      4
2      2
3      3
4      4
      ..
995    3
996    3
997    1
998    6
999    3
Name: month, Length: 1000, dtype: int32

# 
Week

In [20]:
df["week"] = df["defect_date"].dt.isocalendar().week
df["week"] 

0      23
1      17
2       7
3      13
4      17
       ..
995     9
996    12
997     3
998    25
999    12
Name: week, Length: 1000, dtype: UInt32

# 
Day of week 

In [21]:
df["day_of_week"] = df["defect_date"].dt.day_name()
df["day_of_week"] 

0      Thursday
1        Friday
2      Thursday
3      Thursday
4        Friday
         ...   
995      Friday
996    Thursday
997     Tuesday
998      Friday
999    Saturday
Name: day_of_week, Length: 1000, dtype: object

# Cleaned dataset 

In [22]:
df.to_csv("defects_data_clean.csv", index=False)

print(df.head())

   defect_id  product_id defect_type defect_date defect_location  severity  \
0          1          15  structural  2024-06-06       Component     Minor   
1          2           6  functional  2024-04-26       Component     Minor   
2          3          84  structural  2024-02-15        Internal     Minor   
3          4          10  functional  2024-03-28        Internal  Critical   
4          5          14    cosmetic  2024-04-26       Component     Minor   

   inspection_method  repair_cost  month  week day_of_week  
0  Visual Inspection       245.47      6    23    Thursday  
1  Visual Inspection        26.87      4    17      Friday  
2  Automated Testing       835.81      2     7    Thursday  
3  Automated Testing       444.47      3    13    Thursday  
4     Manual Testing       823.64      4    17      Friday  
